In [1]:
# Berechnungen und Plotting
import numpy as np
from scipy import odr
import scipy.optimize
import scipy.constants as c
from scipy.optimize import curve_fit
from scipy.stats import chi2
from scipy.stats import poisson
from scipy.integrate import quad
from scipy.signal import find_peaks
from scipy.signal import argrelextrema, argrelmin, argrelmax
from scipy.special import factorial

In [ ]:
class Base():                                                    # Class name
    def __init__(self, name="Default", purpose="communication"): # Constructor gets called when instance of class is created
        self.name = name                                         # Attribute of instance "self" called "name" will be set to value of variable "name"
        self.purpose = purpose                                   # Attribute set
        self._setText("Class {name} for {purpose}.")             # Call method on this instance "self"

    def _setText(self, text):                                    # Method definition, first argument always is "self" refering to this instance
        self._text = text                                        # Attribute set, underscore prefix indicates internal usage

    def help(self):                                              # Method definition, first argument always is "self" refering to this instance
        print(self._text.format(name=self.name, purpose=self.purpose))  # self._text ist grade noch ein string, der auf zwei Inhalte {name} und {purpose} wartet, mit .format werden diese gegeben


base = Base("GPIB", "hardware communication")                    # Instance of class created and constructor called with arguments. Inside the class the self reference is used for this
base._text
# base.help()  

'Class {name} for {purpose}.'

In [ ]:
from pathlib import Path
import csv

class LIAmp:
    def __init__(self):
        # Attribute im Objekt initialisieren
        self.qx = 1
        self.qy = 2

    def read(self):
        # Greift auf die eigenen Attribute zu
        return self.qx, self.qy

    def am_phi(self):
        A = self.qx + 2
        phi = self.qy + 2
        return A, phi


class DataWriter(LIAmp):
    def __init__(self, path):
        # WICHTIG: init der Elternklasse (LIAmp) aufrufen!
        super().__init__()
        self.name = path
        self._path = Path.cwd()

    def createfile(self):
        with open(self._path / f"{self.name}.csv", 'w', newline='') as csvfile:
            fieldnames = ['QX', 'QY', 'A', 'phi']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()

    def writeData(self):
        qx, qy = self.read()         # Funktioniert jetzt problemlos!
        A, phi = self.am_phi()       # Funktioniert ebenfalls!
        
        with open(self._path / f"{self.name}.csv", 'a', newline='') as csvfile:
            fieldnames = ['QX', 'QY', 'A', 'phi']
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writerow({'QX': qx, 'QY': qy, 'A': A, 'phi': phi})

# Nutzung:
data = DataWriter('Messung1')
data.createfile()
data.writeData()

In [34]:
omega=100*2*np.pi  # Wechselspannungsfrequenz
l=2*1e-2 #Länge des Reeds
b =5*1e-3 # Breite des Reeds
d =200*1e-6 # μm Dicke des Reeds
g_a = 100*1e-6  # Abstand der Anregungselektrode zum Reed
g_d = 100*1e-6  # Abstand der Detektionselektrode zum Reed
D_a = D_d = 3*1e-3 # Durchmesser der Elektroden
U_a = U_0 = 10 # V Amplitude der Anregungsspannung
U_B = 108  # V Batteriespannung (Gleichspannung)
E = 98*1e9 # GN/m2 Elastizit¨atsmodul von Messing (Pl¨attchenmaterial)
rho = 8.5 * 1e3 # kg/m3 Dichte von Messing
C_L = 300 # pF Parasit¨are Leitungskapazit¨at
R = 300 # MΩ Zus¨atzlicher Widerstand
C_a=c.epsilon_0*l*b/g_a
C_d=c.epsilon_0*l*b/g_d

F=1/4*(C_a*U_0**2/g_a)*(2) # cos_max=1
psi_xl=4*l**3/(E*d**3*b)*F
print(psi_xl)
U_d=U_B*psi_xl/g_d*C_d/C_L
print(U_d) 

3.6139542117551045e-08
1.1519506569392282e-15


In [ ]:
alpha_n=np.array([1.424987,0.992249,1.000198])
n=np.array([0.0,1.0,2.0])
freq_n=alpha_n*(2*n+1)**2*np.pi*d/(16*np.sqrt(3)*l**2)*np.sqrt(E/rho)   
print(freq_n)
print(freq_n[1]/freq_n[0])
print(freq_n[2]/freq_n[0])

freq_n=[10,10,10]
for freq in freq_n:

    freq_start=0.8*freq
    freq_stop=1.2*freq
    freq_array=np.linspace(freq_start,freq_stop,2)
    print(freq_array)

[ 274.25448869 1718.72352471 4812.47883447]
6.26689296112877
17.54749341572941
[ 8. 12.]
[ 8. 12.]
[ 8. 12.]


Peak Finder

In [2]:
intensity=[]
# prominence=underground noise
# distance= minimal distance between peaks
peak_indices, _ = scipy.signal.find_peaks(intensity,prominence=1,distance=1)
# information stored in properties can be specified with .find_peaks(desired information commands)
peak_indices, properties = scipy.signal.find_peaks(intensity,prominence=1)
# wenn prominence aktiviert ist, speichert find_peaks die linken und rechten Basen des peaks
left_bases = properties['left_bases']
right_bases = properties['right_bases']
# slicing: arrays slicen, indices ist dann ein array das von start:end geht
indices = slice(peak_indices[1] - 2, peak_indices[2] + 3)

IndexError: index 1 is out of bounds for axis 0 with size 0

Curvefit

In [ ]:
# Fit function with parameters
def gaussian(x,A,mu,sig,c):
    return A/np.sqrt(2*np.pi)*np.exp(1/2*(x-mu)**2/sig**2)+c
# Startparameter
mu_start=10
A_start=10
c_start=1
# Grenzen anhand derer p0 variiert werden darf
untere_grenzen = [0, mu_start - 0.1, 0.001, 0]
obere_grenzen =  [np.inf, mu_start + 0.1, 0.5, np.inf]

popt, pcov = curve_fit(gaussian, x, y, p0=[A_start, mu_start, 0.1, c_start], bounds=(untere_grenzen, obere_grenzen)) # Hier zwingst du ihn auf den richtigen Peak!sigma=Delta_intensityarray, absolute_sigma=True
